# Scenario 6 Workbench: Structured Data Extraction

Welcome to the hands-on offline lab for **Scenario 6**. Read [`SCENARIO_GUIDE.md`](SCENARIO_GUIDE.md) first for schema nullability rules, Extended Thinking constraints, and 6 practice questions.

### What You Will Observe in This Lab
This notebook runs [`simulation.py`](simulation.py) offline so you can watch `StructuredExtractionPipeline`:
1. Recover from a fixable math error (`subtotal + tax != total`) using a bounded retry.
2. Abort retries immediately when a required field is physically absent from the source receipt, preventing hallucination.
3. Build valid forced-tool requests (without `thinking` or `temperature`), validate nullable schemas, and route extractions by confidence.

**Memory Hook:** *"Schema for shape, code for truth, null for missing, Batches for backlog, stratified audits for trust."*

In [ ]:
# Step 1: Environment Setup & Simulation Import
import sys
from pathlib import Path

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scenarios.scenario_6_structured_data_extraction.simulation import *

print(' Scenario Workbench Initialized: Scenario 6: Structured Data Extraction Pipeline')
print(' Zero live API calls, 100% offline determinism')


### Step 2: Recover a Fixable Math Error via Closed-Loop Retry
Simulate a first extraction where `50 + 5 != 50`, then watch the second turn correct `total` to `55.0` and succeed with `1` retry.

In [ ]:
# Test Math Error Recovery via Retry
pipeline = StructuredExtractionPipeline(random_seed=42)
doc_text = "Subtotal: $50, Tax: $5, Total: $55, Tax ID: TX-1"
turns = [
    {"subtotal": 50.0, "tax": 5.0, "total": 50.0, "tax_id": "TX-1"},
    {"subtotal": 50.0, "tax": 5.0, "total": 55.0, "tax_id": "TX-1"}
]
outcome = pipeline.process_extraction("d1", doc_text, turns, confidence_score=0.96)
print("Extraction Status:", outcome.status)
print("Retries Used:", outcome.retries_used)
print("Routing Tier:", outcome.routing_tier)
assert outcome.status == "SUCCESS"
assert outcome.retries_used == 1



### Step 3: Fast-Fail When Source Data Is Physically Missing
Test a cash diner receipt that has no Tax ID anywhere in the text. Confirm the pipeline aborts immediately (`retries_used == 0`) and routes to `HUMAN_TRIAGE` instead of retrying into a hallucination.

In [ ]:
# Test Unrecoverable Missing Fact Aborts Retry Loop Immediately
pipeline = StructuredExtractionPipeline(random_seed=42)
doc_missing = "Simple receipt: Total $10. Paid in cash at diner counter."
turns_missing = [{"subtotal": 10.0, "tax": 0.0, "total": 10.0, "tax_id": None}]
outcome_missing = pipeline.process_extraction("d2", doc_missing, turns_missing, confidence_score=0.95)
print("Missing Fact Status:", outcome_missing.status)
print("Retries Used:", outcome_missing.retries_used)
assert outcome_missing.status == "FAILED_UNRECOVERABLE"
assert outcome_missing.retries_used == 0
assert outcome_missing.routing_tier == ExtractionRoutingTier.HUMAN_TRIAGE



### Step 4: Inspect Forced-Tool Requests, Nullable Schemas & Routing
Check that `build_extraction_request` sets forced `tool_choice` with no `thinking` key, `validate_against_schema` accepts `null` for `tax_id`, and `route_by_confidence` routes low-confidence outputs to human triage.

In [ ]:
req = build_extraction_request("Invoice text")
assert req["tool_choice"]["type"] == "tool" and "thinking" not in req
ok = {"vendor": "A", "total": 10.0, "tax_id": None, "expense_category": "unclear"}
assert validate_against_schema(ok) == []
assert is_retry_eligible(False, True, 0) and not is_retry_eligible(False, False, 0)
pipe = StructuredExtractionPipeline(random_seed=1)
print("Routing 0.5:", pipe.route_by_confidence(0.5)[0].value)


### Lab Complete
All offline checks passed. For full unit tests, run `uv run pytest scenarios/scenario_6_structured_data_extraction/test_scenario_6_simulation.py` in WSL Ubuntu-26.04.